# Sprint 2 — Processamento de Linguagem Natural
## Digital Twin Forzy | FIAP Challenge × Forzy / Promon

**Objetivo:** Implementar busca semântica sobre os ativos industriais do Digital Twin utilizando embeddings multilíngues e índice FAISS.

**Entregáveis cobertos:**
- ✅ Entregável 1 — Representação Vetorial dos Ativos
- ✅ Indexação FAISS
- ✅ Entregável 2 — Módulo de Consulta Semântica
- ✅ Tratamento de Sinônimos
- ✅ Avaliação (Precision@K / Recall@K)
- ✅ Entregável 3 — Enriquecimento Textual

---

### Dados de origem
Todos os dados de ativos e alertas são provenientes de `src/services/api.ts` (Sprint 1).
Nenhuma informação fictícia foi criada — apenas os dados já existentes foram reutilizados.

### Dependências
```bash
pip install sentence-transformers faiss-cpu numpy pandas matplotlib seaborn ipython
```

---
## Seção 0 — Instalação e Imports

In [1]:
# Instalar dependências (executar apenas uma vez)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'sentence-transformers', 'faiss-cpu', 'numpy', 'pandas',
    'matplotlib', 'seaborn', '--quiet'])
print('✅ Dependências instaladas com sucesso.')

✅ Dependências instaladas com sucesso.


In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import faiss
from sentence_transformers import SentenceTransformer

# Reprodutibilidade
np.random.seed(42)

# Diretório de saída dos artefatos
ARTIFACTS_DIR = Path('.')
ARTIFACTS_DIR.mkdir(exist_ok=True)

print('✅ Imports OK')
print(f'   faiss version : {faiss.__version__}')
print(f'   numpy version : {np.__version__}')
print(f'   pandas version: {pd.__version__}')

---
## Seção 1 — Dados dos Ativos (Sprint 1)

Os dados abaixo replicam fielmente o conteúdo de `src/services/api.ts`.
Sensores usam `baseValue` e `threshold = baseValue × 1.2` exatamente como codificado em `generateSensorData()`.

In [ ]:
# ============================================================
# DADOS DOS ATIVOS — origem: src/services/api.ts (Sprint 1)
# ============================================================

ASSETS = [
    {
        'id': 'motor-001',
        'name': 'Main Production Motor',
        'type': 'motor',
        'location': 'Hall A - Station 1',
        'status': 'normal',
        'uptime': 99.8,
        'lastMaintenance': '2026-03-15',
        'nextMaintenance': '2026-06-15',
        'anomalyScore': 12,
        'x': 1, 'y': 1,
        'sensors': [
            {'name': 'Voltage',     'unit': 'V',    'value': 380.0, 'min': 266.0, 'max': 494.0, 'threshold': 456.0, 'status': 'normal'},
            {'name': 'Current',     'unit': 'A',    'value': 45.0,  'min': 31.5,  'max': 58.5,  'threshold': 54.0,  'status': 'normal'},
            {'name': 'Temperature', 'unit': '°C',   'value': 65.0,  'min': 45.5,  'max': 84.5,  'threshold': 78.0,  'status': 'normal'},
            {'name': 'RPM',         'unit': 'rpm',  'value': 1480,  'min': 1036,  'max': 1924,  'threshold': 1776,  'status': 'normal'},
            {'name': 'Vibration',   'unit': 'mm/s', 'value': 2.5,   'min': 1.75,  'max': 3.25,  'threshold': 3.0,   'status': 'normal'},
        ]
    },
    {
        'id': 'motor-002',
        'name': 'Backup Motor',
        'type': 'motor',
        'location': 'Hall A - Station 2',
        'status': 'warning',
        'uptime': 95.2,
        'lastMaintenance': '2026-01-20',
        'nextMaintenance': '2026-04-20',
        'anomalyScore': 58,
        'x': 3, 'y': 1,
        'sensors': [
            {'name': 'Voltage',     'unit': 'V',    'value': 380.0, 'min': 266.0, 'max': 494.0, 'threshold': 456.0, 'status': 'normal'},
            {'name': 'Current',     'unit': 'A',    'value': 52.0,  'min': 36.4,  'max': 67.6,  'threshold': 62.4,  'status': 'normal'},
            {'name': 'Temperature', 'unit': '°C',   'value': 72.0,  'min': 50.4,  'max': 93.6,  'threshold': 86.4,  'status': 'warning'},
            {'name': 'RPM',         'unit': 'rpm',  'value': 1470,  'min': 1029,  'max': 1911,  'threshold': 1764,  'status': 'normal'},
            {'name': 'Vibration',   'unit': 'mm/s', 'value': 4.2,   'min': 2.94,  'max': 5.46,  'threshold': 5.04,  'status': 'warning'},
        ]
    },
    {
        'id': 'pump-001',
        'name': 'Coolant Circulation Pump',
        'type': 'pump',
        'location': 'Hall B - Cooling System',
        'status': 'normal',
        'uptime': 99.5,
        'lastMaintenance': '2026-02-28',
        'nextMaintenance': '2026-05-28',
        'anomalyScore': 8,
        'x': 1, 'y': 3,
        'sensors': [
            {'name': 'Pressure',     'unit': 'bar',   'value': 2.5,  'min': 1.75, 'max': 3.25, 'threshold': 3.0,  'status': 'normal'},
            {'name': 'Flow Rate',    'unit': 'L/min', 'value': 150,  'min': 105,  'max': 195,  'threshold': 180,  'status': 'normal'},
            {'name': 'Temperature',  'unit': '°C',    'value': 45.0, 'min': 31.5, 'max': 58.5, 'threshold': 54.0, 'status': 'normal'},
            {'name': 'Power',        'unit': 'kW',    'value': 5.0,  'min': 3.5,  'max': 6.5,  'threshold': 6.0,  'status': 'normal'},
        ]
    },
    {
        'id': 'compressor-001',
        'name': 'Air Compressor',
        'type': 'compressor',
        'location': 'Hall C - Pneumatics',
        'status': 'critical',
        'uptime': 78.3,
        'lastMaintenance': '2025-12-10',
        'nextMaintenance': '2026-03-10',
        'anomalyScore': 85,
        'x': 3, 'y': 3,
        'sensors': [
            {'name': 'Outlet Pressure', 'unit': 'bar', 'value': 7.0,  'min': 4.9,  'max': 9.1,   'threshold': 8.4,  'status': 'normal'},
            {'name': 'Inlet Temp',      'unit': '°C',  'value': 28.0, 'min': 19.6, 'max': 36.4,  'threshold': 33.6, 'status': 'normal'},
            {'name': 'Outlet Temp',     'unit': '°C',  'value': 82.0, 'min': 57.4, 'max': 106.6, 'threshold': 98.4, 'status': 'warning'},
            {'name': 'Motor Current',   'unit': 'A',   'value': 35.0, 'min': 24.5, 'max': 45.5,  'threshold': 42.0, 'status': 'normal'},
            {'name': 'Vibration',       'unit': 'mm/s','value': 6.8,  'min': 4.76, 'max': 8.84,  'threshold': 8.16, 'status': 'warning'},
        ]
    },
    {
        'id': 'fan-001',
        'name': 'Ventilation Fan',
        'type': 'fan',
        'location': 'Hall A - Roof',
        'status': 'normal',
        'uptime': 99.2,
        'lastMaintenance': '2026-03-01',
        'nextMaintenance': '2026-06-01',
        'anomalyScore': 15,
        'x': 2, 'y': 2,
        'sensors': [
            {'name': 'Speed',         'unit': '%',    'value': 85.0, 'min': 59.5, 'max': 110.5, 'threshold': 102.0, 'status': 'normal'},
            {'name': 'Motor Current', 'unit': 'A',    'value': 8.0,  'min': 5.6,  'max': 10.4,  'threshold': 9.6,   'status': 'normal'},
            {'name': 'Bearing Temp',  'unit': '°C',   'value': 55.0, 'min': 38.5, 'max': 71.5,  'threshold': 66.0,  'status': 'normal'},
            {'name': 'Vibration',     'unit': 'mm/s', 'value': 1.2,  'min': 0.84, 'max': 1.56,  'threshold': 1.44,  'status': 'normal'},
        ]
    },
]

# Alertas com mlInsight (origem: src/services/api.ts)
ALERTS = [
    {
        'id': 'alert-001',
        'assetId': 'compressor-001',
        'assetName': 'Air Compressor',
        'severity': 'critical',
        'title': 'High Vibration Detected',
        'description': 'Vibration levels have exceeded safe thresholds. Immediate maintenance required.',
        'mlInsight': 'ML Model predicts bearing failure within 7 days with 89% confidence.',
        'resolved': False,
    },
    {
        'id': 'alert-002',
        'assetId': 'motor-002',
        'assetName': 'Backup Motor',
        'severity': 'warning',
        'title': 'Elevated Temperature',
        'description': 'Motor temperature is above normal operating range.',
        'mlInsight': 'Temperature trend suggests cooling system degradation. Recommend coolant check.',
        'resolved': False,
    },
    {
        'id': 'alert-003',
        'assetId': 'motor-001',
        'assetName': 'Main Production Motor',
        'severity': 'info',
        'title': 'Scheduled Maintenance Due',
        'description': 'Preventive maintenance is scheduled for June 15, 2026.',
        'mlInsight': None,
        'resolved': False,
    },
]

print(f'✅ Dados carregados: {len(ASSETS)} ativos, {len(ALERTS)} alertas.')
for a in ASSETS:
    print(f"   {a['id']:20s} | status: {a['status']:8s} | anomalyScore: {a['anomalyScore']}")

---
## Seção 2 — Geração de TAGs e Enriquecimento de Campos

A análise da Sprint 1 identificou que o campo `asset.id` funciona como TAG mas não está
explicitamente mapeado para um código industrial padronizado. Aqui geramos o campo `tag`
seguindo o padrão `{PREFIXO}-{NÚMERO}` (`MTR`, `BOM`, `CMP`, `VEN`).

In [ ]:
# ============================================================
# GERAÇÃO DE TAGs PADRONIZADAS
# ============================================================

TYPE_PREFIX = {
    'motor':       'MTR',
    'pump':        'BOM',
    'compressor':  'CMP',
    'fan':         'VEN',
}

TYPE_PT = {
    'motor':       'Motor',
    'pump':        'Bomba',
    'compressor':  'Compressor',
    'fan':         'Ventilador',
}

def generate_tag(asset_id: str, asset_type: str) -> str:
    """Converte asset.id para TAG padronizada industrial.
    Exemplos: motor-001 → MTR-001 | pump-001 → BOM-001
    """
    number_match = re.search(r'(\d+)$', asset_id)
    number = number_match.group(1) if number_match else '001'
    prefix = TYPE_PREFIX.get(asset_type, 'EQP')
    return f'{prefix}-{number}'

# Enricher: dados de placa simulados (em produção viriam do OCR / cadastro)
PLATE_DATA = {
    'motor-001':      {'manufacturer': 'WEG',      'model': 'W22 IE3 132M',  'voltage': '380V', 'power': '45kW',  'poles': 4},
    'motor-002':      {'manufacturer': 'ABB',      'model': 'IE3-132M',      'voltage': '380V', 'power': '52kW',  'poles': 4},
    'pump-001':       {'manufacturer': 'KSB',      'model': 'Etanorm 80-250','voltage': '380V', 'power': '5kW',   'poles': 2},
    'compressor-001': {'manufacturer': 'Schulz',   'model': 'MSV-15 Max',    'voltage': '380V', 'power': '15kW',  'poles': 2},
    'fan-001':        {'manufacturer': 'Multivac', 'model': 'EVT-250',       'voltage': '220V', 'power': '2.2kW', 'poles': 4},
}

# Adicionar campos derivados a cada ativo
for a in ASSETS:
    a['tag']       = generate_tag(a['id'], a['type'])
    a['type_pt']   = TYPE_PT.get(a['type'], a['type'].capitalize())
    plate          = PLATE_DATA.get(a['id'], {})
    a['manufacturer'] = plate.get('manufacturer', 'N/A')
    a['model']        = plate.get('model', 'N/A')
    a['voltage']      = plate.get('voltage', 'N/A')
    a['power']        = plate.get('power', 'N/A')
    a['poles']        = plate.get('poles', None)

print('✅ TAGs geradas:')
for a in ASSETS:
    print(f"   {a['id']:20s} → TAG: {a['tag']}  |  {a['manufacturer']:8s} {a['model']}")

---
## Seção 3 — Entregável 3: Templates de Enriquecimento Textual

Transformamos os dados estruturados em descrições de linguagem natural seguindo as
regras documentadas em `sprint2_pln_templates.md`.

In [ ]:
# ============================================================
# TEMPLATES DE LINGUAGEM NATURAL
# ============================================================

def sensor_status_label(status: str) -> str:
    return {'normal': 'normal', 'warning': 'alerta', 'critical': 'crítico'}.get(status, status)

def template_base(a: dict) -> str:
    """Descrição de identidade do ativo."""
    loc_pt = a['location'].replace('Station', 'Estação').replace('Cooling System', 'Sistema de Resfriamento')\
                          .replace('Pneumatics', 'Pneumática').replace('Roof', 'Cobertura')
    return (
        f"{a['type_pt']} {a['tag']} ({a['name']}), "
        f"localizado em {loc_pt}, "
        f"fabricante {a['manufacturer']}, modelo {a['model']}. "
        f"Tensão de operação {a['voltage']}, potência {a['power']}"
        + (f", {a['poles']} polos" if a['poles'] else '') + "."
    )

def template_status(a: dict) -> str:
    """Descrição do estado operacional com foco nos sensores mais críticos."""
    tag   = a['tag']
    score = a['anomalyScore']
    
    # Sensores com problema
    anomalous = [s for s in a['sensors'] if s['status'] != 'normal']
    
    if a['status'] == 'normal':
        # Buscar temperatura e vibração para exemplificar normalidade
        temp  = next((s for s in a['sensors'] if 'temp' in s['name'].lower()), None)
        vib   = next((s for s in a['sensors'] if 'vibr' in s['name'].lower() or 'vibra' in s['name'].lower()), None)
        curr  = next((s for s in a['sensors'] if 'current' in s['name'].lower()), None)
        parts = []
        if temp:  parts.append(f"temperatura: {temp['value']}{temp['unit']} (OK)")
        if vib:   parts.append(f"vibração: {vib['value']}{vib['unit']} (OK)")
        if curr:  parts.append(f"corrente: {curr['value']}{curr['unit']} (OK)")
        sensor_str = ', '.join(parts) if parts else 'todos os parâmetros normais'
        return (
            f"{tag} operando dentro dos parâmetros normais. {sensor_str.capitalize()}. "
            f"Disponibilidade: {a['uptime']}%. "
            f"Próxima manutenção: {a['nextMaintenance']}."
        )
    
    elif a['status'] == 'warning':
        sensor_str = ', '.join(
            f"{s['name']} {s['value']}{s['unit']} (limite: {s['threshold']}{s['unit']})"
            for s in anomalous
        ) if anomalous else 'parâmetros acima do recomendado'
        return (
            f"{tag} em estado de alerta. {sensor_str.capitalize()}. "
            f"Monitoramento intensificado recomendado. Índice de anomalia: {score}%. "
            f"Disponibilidade: {a['uptime']}%."
        )
    
    elif a['status'] == 'critical':
        sensor_str = ', '.join(
            f"{s['name']} {s['value']}{s['unit']} (limiar: {s['threshold']}{s['unit']})"
            for s in anomalous
        ) if anomalous else 'múltiplos parâmetros críticos'
        return (
            f"{tag} em estado crítico. {sensor_str.capitalize()}. "
            f"Parada para inspeção imediata recomendada. Índice de anomalia: {score}%. "
            f"Risco de falha detectado. Disponibilidade: {a['uptime']}%."
        )
    
    elif a['status'] == 'offline':
        return (
            f"{tag} offline — sem leitura de sensores. "
            f"Verificar conectividade e estado físico do equipamento."
        )
    
    return f"{tag} — estado desconhecido."

def template_sensores(a: dict) -> str:
    """Leituras de sensores em linguagem natural."""
    tag     = a['tag']
    leituras = ', '.join(
        f"{s['name'].lower()} {s['value']}{s['unit']} ({sensor_status_label(s['status'])})"
        for s in a['sensors']
    )
    return f"Telemetria de {tag}: {leituras}."

def template_tipo_extra(a: dict) -> str:
    """Termos adicionais para melhorar recall semântico."""
    tipo = a['type']
    extras = {
        'motor':      'motor elétrico trifásico assíncrono industrial ativo máquina equipamento',
        'pump':       'bomba centrífuga equipamento bombeamento fluido ativo máquina',
        'compressor': 'compressor de ar pneumático equipamento ativo máquina compressão',
        'fan':        'ventilador exaustor extrator de ar equipamento ativo máquina ventilação',
    }
    return f"Tipo: {extras.get(tipo, tipo)}."

def template_completo(a: dict) -> str:
    """Texto completo para embedding: base + status + sensores + extras."""
    return ' '.join([
        template_base(a),
        template_status(a),
        template_sensores(a),
        template_tipo_extra(a),
    ])

# Testar os templates
print('=' * 70)
for a in ASSETS:
    print(f"\n{'─'*60}")
    print(f"ATIVO: {a['tag']} — {a['name']} [{a['status'].upper()}]")
    print(f"\n[BASE]\n{template_base(a)}")
    print(f"\n[STATUS]\n{template_status(a)}")
    print(f"\n[SENSORES]\n{template_sensores(a)}")
    print(f"\n[TIPO EXTRA]\n{template_tipo_extra(a)}")

In [ ]:
# Adicionar o texto completo a cada ativo
for a in ASSETS:
    a['text_embedding'] = template_completo(a)

print('✅ Textos para embedding gerados:')
for a in ASSETS:
    print(f"\n[{a['tag']}] ({len(a['text_embedding'])} chars)")
    print(a['text_embedding'][:200] + '...')

---
## Seção 4 — Entregável 1: Modelo de Embedding

### Justificativa técnica: `paraphrase-multilingual-MiniLM-L12-v2`

| Critério | Justificativa |
|---|---|
| **Suporte pt/BR** | Treinado em 50+ idiomas incluindo português com dados de alta qualidade |
| **Suporte en** | Inglês é um dos idiomas primários do treinamento |
| **Domínio industrial** | Sentence-Transformers é agnóstico ao domínio; termos técnicos (WEG, RPM, kW, bar) são preservados como tokens |
| **Eficiência computacional** | MiniLM com 12 camadas e 384 dimensões — 5× mais rápido que `all-mpnet-base-v2` sem perda significativa de qualidade para busca |
| **Compatibilidade** | Nativo SentenceTransformers; exportável para ONNX; integração direta com FAISS |
| **Alternativa OpenAI** | `text-embedding-3-small` (1536 dims) seria superior em qualidade mas requer API key e custo por token — não adequado para prototipagem offline |
| **Tamanho** | ~118 MB — adequado para ambiente acadêmico/demo |

**Dimensão de saída:** 384 dimensões por vetor

In [ ]:
# ============================================================
# CARREGAMENTO DO MODELO DE EMBEDDING
# ============================================================

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f'⏳ Carregando modelo: {MODEL_NAME}')
print('   (download ~118 MB na primeira execução)')

model = SentenceTransformer(MODEL_NAME)

EMBEDDING_DIM = model.get_sentence_embedding_dimension()

print(f'\n✅ Modelo carregado!')
print(f'   Nome         : {MODEL_NAME}')
print(f'   Dimensão     : {EMBEDDING_DIM}')
print(f'   Idiomas      : 50+ (pt, en, es, de, fr, ...)')
print(f'   Arquitetura  : MiniLM-L12 (12 camadas transformer)')
print(f'   Pooling      : mean pooling com normalização L2')

# Teste rápido de qualidade semântica
test_pairs = [
    ("motor com temperatura alta", "equipamento com calor excessivo"),
    ("compressor crítico", "bomba normal"),
    ("vibração elevada no ativo", "high vibration in equipment"),
]

print(f'\n── Teste de similaridade semântica:')
for s1, s2 in test_pairs:
    e1 = model.encode([s1])
    e2 = model.encode([s2])
    sim = float(np.dot(e1, e2.T) / (np.linalg.norm(e1) * np.linalg.norm(e2)))
    print(f'   "{s1}" ↔ "{s2}" → sim={sim:.3f}')

---
## Seção 5 — Entregável 1: Geração de Embeddings e Indexação FAISS

### Estratégia de indexação

| Parâmetro | Valor | Justificativa |
|---|---|---|
| Tipo do índice | `IndexFlatIP` | Busca exata por produto interno (= coseno após norm L2) — adequado para N=5 ativos; sem aproximação |
| Dimensão | 384 | Saída do `paraphrase-multilingual-MiniLM-L12-v2` |
| Métrica | Inner Product (produto interno) | Equivalente à similaridade coseno quando vetores estão normalizados L2 |
| Normalização | L2 norm aplicada antes de inserir | Garante que IP = coseno — métrica ideal para similaridade textual |
| Serialização | `faiss.write_index()` + JSON de metadados | Permite reutilização sem re-treinar |

**Por que não `IndexIVFFlat`?** Com apenas 5 ativos, o overhead de clustering (necessita ≥ 39× N vetores para treinar `nlist` clusters) não se justifica. `IndexFlatIP` é O(N) e perfeitamente eficiente.

In [ ]:
# ============================================================
# PIPELINE: TEXTO → EMBEDDING → NORMALIZAÇÃO → FAISS
# ============================================================

# 1. Extrair textos
texts    = [a['text_embedding'] for a in ASSETS]
ids      = [a['id'] for a in ASSETS]
tags     = [a['tag'] for a in ASSETS]

print('⏳ Gerando embeddings...')
embeddings = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,   # normalização L2 embutida
    convert_to_numpy=True
).astype(np.float32)

print(f'\n✅ Embeddings gerados: shape = {embeddings.shape}')
print(f'   N vetores  : {embeddings.shape[0]}')
print(f'   Dimensão   : {embeddings.shape[1]}')
print(f'   Dtype      : {embeddings.dtype}')
print(f'   Norm L2[0] : {np.linalg.norm(embeddings[0]):.4f} (deve ser ≈ 1.0)')

In [ ]:
# ============================================================
# CONSTRUÇÃO DO ÍNDICE FAISS
# ============================================================

# IndexFlatIP = produto interno → equivale a coseno quando vetores são normalizados L2
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings)

print(f'✅ Índice FAISS criado!')
print(f'   Tipo         : IndexFlatIP (Inner Product = coseno após norm L2)')
print(f'   Dimensão     : {index.d}')
print(f'   Total vetores: {index.ntotal}')
print(f'   É treinado   : {index.is_trained}')

# Metadados dos ativos para recuperação
METADATA = [
    {
        'faiss_idx'   : i,
        'id'          : a['id'],
        'tag'         : a['tag'],
        'name'        : a['name'],
        'type'        : a['type'],
        'type_pt'     : a['type_pt'],
        'location'    : a['location'],
        'status'      : a['status'],
        'uptime'      : a['uptime'],
        'anomalyScore': a['anomalyScore'],
        'manufacturer': a['manufacturer'],
        'model'       : a['model'],
        'voltage'     : a['voltage'],
        'power'       : a['power'],
    }
    for i, a in enumerate(ASSETS)
]

# Serialização
INDEX_PATH    = ARTIFACTS_DIR / 'faiss_assets.index'
METADATA_PATH = ARTIFACTS_DIR / 'assets_metadata.json'

faiss.write_index(index, str(INDEX_PATH))
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(METADATA, f, ensure_ascii=False, indent=2)

print(f'\n✅ Artefatos salvos:')
print(f'   📦 {INDEX_PATH}   ({INDEX_PATH.stat().st_size} bytes)')
print(f'   📋 {METADATA_PATH} ({METADATA_PATH.stat().st_size} bytes)')

---
## Seção 6 — Tratamento de Sinônimos (Expansão Semântica)

In [ ]:
# ============================================================
# MAPA DE SINÔNIMOS INDUSTRIAIS
# ============================================================

SYNONYM_MAP = {
    # Tipos de equipamento
    'motor':       ['motor', 'máquina', 'equipamento', 'ativo', 'motor elétrico',
                    'motor trifásico', 'motor assíncrono', 'machine', 'equipment'],
    'bomba':       ['bomba', 'pump', 'bomba centrífuga', 'circulador'],
    'compressor':  ['compressor', 'compressor de ar', 'air compressor'],
    'ventilador':  ['ventilador', 'fan', 'exaustor', 'extrator'],

    # Localização
    'hall':        ['hall', 'área', 'setor', 'pavilhão', 'galpão', 'zone', 'area', 'sector'],
    'estação':     ['estação', 'station', 'ponto', 'posição', 'posto'],
    'localização': ['localização', 'local', 'localidade', 'posição', 'where', 'location'],
    'cobertura':   ['cobertura', 'teto', 'roof', 'telhado'],

    # Estado
    'normal':      ['normal', 'ok', 'operacional', 'funcionando', 'healthy', 'good'],
    'alerta':      ['alerta', 'aviso', 'warning', 'atenção', 'monitorar'],
    'crítico':     ['crítico', 'critical', 'falha', 'failure', 'quebrado', 'parado'],

    # Sensores
    'temperatura': ['temperatura', 'temperature', 'temp', 'calor', 'thermal', 'hot', 'quente'],
    'vibração':    ['vibração', 'vibration', 'vib', 'vibrating', 'vibra', 'oscilação'],
    'corrente':    ['corrente', 'current', 'amperagem', 'ampere', 'amp', 'carga elétrica'],
    'tensão':      ['tensão', 'voltage', 'volt', 'voltagem', '220v', '380v', '440v'],
    'rotação':     ['rotação', 'rpm', 'velocidade', 'speed', 'giro'],
    'pressão':     ['pressão', 'pressure', 'bar', 'psi'],
    'vazão':       ['vazão', 'flow', 'flow rate', 'fluxo'],
}

# Índice invertido: variante → termo canônico
SYNONYM_INDEX = {}
for canonical, variants in SYNONYM_MAP.items():
    for v in variants:
        SYNONYM_INDEX[v.lower()] = canonical

def expand_query(query: str) -> str:
    """Expande a query substituindo sinônimos pelos termos canônicos.
    
    Estratégia:
    1. Tokeniza a query em palavras
    2. Para cada token, verifica no índice invertido
    3. Se encontrado, adiciona o canonical E mantém o original
    4. Retorna a query enriquecida
    """
    words  = query.lower().split()
    extras = []
    for w in words:
        canonical = SYNONYM_INDEX.get(w)
        if canonical and canonical != w:
            extras.append(canonical)
    expanded = query
    if extras:
        expanded = query + ' ' + ' '.join(set(extras))
    return expanded

# Demonstração da expansão
demo_queries = [
    'equipamento com vibration alto',
    'máquina na área A',
    'ativo com falha',
    'motor quente no setor B',
    'compressor failure',
]
print('✅ Expansão semântica de sinônimos:')
for q in demo_queries:
    expanded = expand_query(q)
    print(f'   Original : "{q}"')
    print(f'   Expandida: "{expanded}"')
    print()

---
## Seção 7 — Entregável 2: Módulo de Consulta Semântica

In [ ]:
# ============================================================
# MOTOR DE BUSCA SEMÂNTICA
# ============================================================

def semantic_search(
    query: str,
    top_k: int = 3,
    use_expansion: bool = True,
    verbose: bool = True
) -> list[dict]:
    """Busca semântica sobre o índice FAISS de ativos.
    
    Parâmetros
    ----------
    query          : Pergunta do operador em linguagem natural
    top_k          : Número de resultados a retornar
    use_expansion  : Aplicar expansão de sinônimos antes do embedding
    verbose        : Imprimir resultado formatado
    
    Retorna
    -------
    Lista de dicts com campos: tag, name, location, status, score, rank
    """
    # 1. Expansão semântica
    query_proc = expand_query(query) if use_expansion else query
    
    # 2. Embedding da query
    q_emb = model.encode(
        [query_proc],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    
    # 3. Busca FAISS (Inner Product = coseno)
    scores, indices = index.search(q_emb, min(top_k, index.ntotal))
    
    # 4. Construir resultado
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        meta = METADATA[idx]
        results.append({
            'rank'         : rank + 1,
            'tag'          : meta['tag'],
            'name'         : meta['name'],
            'type'         : meta['type'],
            'location'     : meta['location'],
            'status'       : meta['status'],
            'anomalyScore' : meta['anomalyScore'],
            'manufacturer' : meta['manufacturer'],
            'model'        : meta['model'],
            'score'        : float(score),
        })
    
    if verbose:
        print(f'\n🔍 Query: "{query}"')
        if use_expansion and query_proc != query:
            print(f'   Expandida: "{query_proc}"')
        print(f'   Top-{top_k} resultados:')
        for r in results:
            status_icon = {'normal': '🟢', 'warning': '🟡', 'critical': '🔴', 'offline': '⚫'}.get(r['status'], '⚪')
            print(f"   [{r['rank']}] {r['tag']:7s} | {r['name']:28s} | {r['location']:25s} | {status_icon} {r['status']:8s} | score={r['score']:.4f}")
    
    return results

# Demonstração com as queries de exemplo do requisito
demo_queries_req = [
    'motor da área 3',
    'equipamento de 220V',
    'motor mais quente',
    'ativo localizado na manutenção',
    'motor trifásico',
]

print('=' * 75)
print('DEMONSTRAÇÃO DAS QUERIES DO REQUISITO')
print('=' * 75)
for q in demo_queries_req:
    semantic_search(q, top_k=3)

---
## Seção 8 — Avaliação: Precision@K e Recall@K

### Conjunto de consultas de teste

20 consultas com ground-truth definido manualmente com base nos dados reais de `api.ts`.

In [ ]:
# ============================================================
# CONJUNTO DE AVALIAÇÃO — 20 CONSULTAS COM GROUND-TRUTH
# ============================================================

# Cada item: (query, set de asset_ids relevantes)
EVAL_SET = [
    # --- Motores ---
    ("motor principal de produção",           {'motor-001'}),
    ("motor de backup reserva",               {'motor-002'}),
    ("motor trifásico Hall A",                {'motor-001', 'motor-002'}),
    ("equipamento WEG na estação 1",          {'motor-001'}),
    ("motor ABB em alerta",                   {'motor-002'}),
    # --- Temperatura / Calor ---
    ("motor com temperatura elevada",         {'motor-002'}),
    ("ativo mais quente",                     {'compressor-001', 'motor-002'}),
    ("equipamento com temperatura acima do limite", {'motor-002', 'compressor-001'}),
    # --- Vibração ---
    ("vibração crítica",                      {'compressor-001'}),
    ("ativo com vibração alta",               {'compressor-001', 'motor-002'}),
    # --- Compressor ---
    ("compressor de ar pneumático",           {'compressor-001'}),
    ("compressor crítico falha",              {'compressor-001'}),
    ("equipamento em estado crítico",         {'compressor-001'}),
    # --- Bomba ---
    ("bomba de resfriamento circulação",      {'pump-001'}),
    ("pump Hall B cooling system",            {'pump-001'}),
    # --- Ventilador ---
    ("ventilador no teto cobertura",          {'fan-001'}),
    ("fan ventilation roof",                  {'fan-001'}),
    # --- Localização ---
    ("equipamentos na área Hall A",           {'motor-001', 'motor-002', 'fan-001'}),
    ("ativo na Hall C pneumática",            {'compressor-001'}),
    # --- Tensão / Elétrico ---
    ("equipamento de 220V ventilador",        {'fan-001'}),
]

print(f'✅ Conjunto de avaliação: {len(EVAL_SET)} consultas')
for i, (q, gt) in enumerate(EVAL_SET, 1):
    print(f'   [{i:2d}] "{q}" → {[g for g in gt]}')

In [ ]:
# ============================================================
# CÁLCULO DE PRECISION@K E RECALL@K
# ============================================================

def evaluate(eval_set: list, k_values: list = [1, 2, 3]) -> pd.DataFrame:
    """Avalia o sistema de busca semântica.
    
    Precision@K = |relevantes retornados em top-K| / K
    Recall@K    = |relevantes retornados em top-K| / |relevantes total|
    """
    rows = []
    
    for query, relevant_ids in eval_set:
        results = semantic_search(query, top_k=max(k_values), verbose=False)
        retrieved_ids = [r['tag'] for r in results]  # usar tag como ID no contexto
        
        # Mapear asset_id → tag para o ground-truth
        relevant_tags = set()
        for aid in relevant_ids:
            meta = next((m for m in METADATA if m['id'] == aid), None)
            if meta:
                relevant_tags.add(meta['tag'])
        
        row = {'query': query, 'relevant': list(relevant_tags), 'retrieved': retrieved_ids}
        
        for k in k_values:
            top_k_ids = set(retrieved_ids[:k])
            hits = len(top_k_ids & relevant_tags)
            row[f'P@{k}'] = hits / k
            row[f'R@{k}'] = hits / len(relevant_tags) if relevant_tags else 0.0
        
        rows.append(row)
    
    return pd.DataFrame(rows)

K_VALUES = [1, 2, 3]
df_eval  = evaluate(EVAL_SET, k_values=K_VALUES)

# Tabela de resultados
display_cols = ['query', 'retrieved'] + [f'P@{k}' for k in K_VALUES] + [f'R@{k}' for k in K_VALUES]
print('\n✅ Tabela de Avaliação')
print('=' * 90)
with pd.option_context('display.max_colwidth', 45, 'display.float_format', '{:.2f}'.format):
    print(df_eval[display_cols].to_string(index=False))

# Médias
print('\n── Médias gerais:')
for k in K_VALUES:
    mp = df_eval[f'P@{k}'].mean()
    mr = df_eval[f'R@{k}'].mean()
    print(f'   Precision@{k} = {mp:.3f}   Recall@{k} = {mr:.3f}')

In [ ]:
# ============================================================
# GRÁFICOS DE AVALIAÇÃO
# ============================================================

sns.set_style('darkgrid')
plt.rcParams.update({'figure.facecolor': '#0f172a', 'axes.facecolor': '#1e293b',
                     'text.color': '#e2e8f0', 'axes.labelcolor': '#e2e8f0',
                     'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
                     'axes.edgecolor': '#334155', 'grid.color': '#334155',
                     'font.family': 'monospace'})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Avaliação Sprint 2 PLN — Busca Semântica FAISS\nForzy Digital Twin | paraphrase-multilingual-MiniLM-L12-v2',
             fontsize=13, fontweight='bold', color='#f1f5f9', y=1.01)

ORANGE  = '#f97316'
BLUE    = '#38bdf8'
GREEN   = '#34d399'
PURPLE  = '#a78bfa'

# ── Gráfico 1: Precision@K médio por K
ax1 = axes[0, 0]
k_labels = [f'P@{k}' for k in K_VALUES]
p_means  = [df_eval[c].mean() for c in k_labels]
bars = ax1.bar(k_labels, p_means, color=[ORANGE, BLUE, GREEN], edgecolor='none', width=0.5)
ax1.set_ylim(0, 1.1)
ax1.set_title('Precision@K Médio', fontweight='bold', color='#f1f5f9')
ax1.set_ylabel('Precision')
for bar, val in zip(bars, p_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='white')

# ── Gráfico 2: Recall@K médio por K
ax2 = axes[0, 1]
r_labels = [f'R@{k}' for k in K_VALUES]
r_means  = [df_eval[c].mean() for c in r_labels]
bars2 = ax2.bar(r_labels, r_means, color=[ORANGE, BLUE, GREEN], edgecolor='none', width=0.5)
ax2.set_ylim(0, 1.1)
ax2.set_title('Recall@K Médio', fontweight='bold', color='#f1f5f9')
ax2.set_ylabel('Recall')
for bar, val in zip(bars2, r_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='white')

# ── Gráfico 3: P@K por consulta (heatmap)
ax3 = axes[1, 0]
pk_matrix = df_eval[[f'P@{k}' for k in K_VALUES]].values
im = ax3.imshow(pk_matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax3.set_xticks(range(len(K_VALUES)))
ax3.set_xticklabels([f'P@{k}' for k in K_VALUES], color='#94a3b8')
ax3.set_yticks(range(len(df_eval)))
ax3.set_yticklabels([q[:30] + '…' if len(q) > 30 else q for q in df_eval['query']], fontsize=7)
ax3.set_title('Precision@K por Consulta (heatmap)', fontweight='bold', color='#f1f5f9')
plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

# ── Gráfico 4: Curva Precision × Recall
ax4 = axes[1, 1]
for i, q in enumerate(df_eval['query']):
    p_vals = [df_eval.loc[i, f'P@{k}'] for k in K_VALUES]
    r_vals = [df_eval.loc[i, f'R@{k}'] for k in K_VALUES]
    ax4.plot(r_vals, p_vals, 'o-', alpha=0.4, linewidth=1, markersize=3, color=BLUE)

# Curva média
p_avg = [df_eval[f'P@{k}'].mean() for k in K_VALUES]
r_avg = [df_eval[f'R@{k}'].mean() for k in K_VALUES]
ax4.plot(r_avg, p_avg, 'o-', linewidth=3, markersize=8, color=ORANGE, label='Média', zorder=5)
for p, r, k in zip(p_avg, r_avg, K_VALUES):
    ax4.annotate(f'K={k}\nP={p:.2f}\nR={r:.2f}', (r, p), textcoords='offset points',
                 xytext=(8, 4), fontsize=7, color=ORANGE)
ax4.set_xlim(-0.05, 1.1)
ax4.set_ylim(-0.05, 1.1)
ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Curva Precision × Recall', fontweight='bold', color='#f1f5f9')
ax4.legend(facecolor='#1e293b', labelcolor='white', edgecolor='#334155')

plt.tight_layout()
FIG_PATH = ARTIFACTS_DIR / 'sprint2_pln_avaliacao.png'
plt.savefig(str(FIG_PATH), dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'\n✅ Gráfico salvo: {FIG_PATH}')

---
## Seção 9 — Análise de Similaridade entre Ativos

In [ ]:
# ============================================================
# MATRIZ DE SIMILARIDADE COSENO ENTRE ATIVOS
# ============================================================

sim_matrix = embeddings @ embeddings.T  # produto interno = coseno (vetores normalizados)
tag_labels = [a['tag'] for a in ASSETS]

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')

im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(tag_labels)))
ax.set_yticks(range(len(tag_labels)))
ax.set_xticklabels(tag_labels, rotation=45, ha='right', color='#94a3b8')
ax.set_yticklabels(tag_labels, color='#94a3b8')

for i in range(len(tag_labels)):
    for j in range(len(tag_labels)):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if sim_matrix[i, j] > 0.5 else '#94a3b8')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Matriz de Similaridade Coseno entre Ativos\n(embeddings paraphrase-multilingual-MiniLM-L12-v2)',
             fontweight='bold', color='#f1f5f9', pad=12)

plt.tight_layout()
SIM_PATH = ARTIFACTS_DIR / 'sprint2_pln_similaridade.png'
plt.savefig(str(SIM_PATH), dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print('\n── Análise de similaridade:')
print('   MTR-001 × MTR-002:', f'{sim_matrix[0,1]:.3f}', '→ Esperado: alta (ambos motores Hall A)')
print('   MTR-001 × CMP-001:', f'{sim_matrix[0,3]:.3f}', '→ Esperado: baixa (tipos diferentes)')
print('   MTR-001 × BOM-001:', f'{sim_matrix[0,2]:.3f}', '→ Esperado: média (ambos normal, elétricos)')
print(f'\n✅ Gráfico salvo: {SIM_PATH}')

---
## Seção 10 — Teste de Carga do Índice Salvo

In [ ]:
# ============================================================
# VERIFICAÇÃO: CARREGAR ÍNDICE DO DISCO E REUTILIZAR
# ============================================================

print('⏳ Carregando índice do disco...')
index_loaded = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata_loaded = json.load(f)

print(f'✅ Índice carregado: {index_loaded.ntotal} vetores, dimensão {index_loaded.d}')

# Verificar consistência com busca ao vivo
q_test = 'compressor crítico com vibração'
q_emb  = model.encode([q_test], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
scores_reload, idxs_reload = index_loaded.search(q_emb, 3)

print(f'\n🔍 Query de validação: "{q_test}"')
for rank, (idx, score) in enumerate(zip(idxs_reload[0], scores_reload[0])):
    m = metadata_loaded[idx]
    print(f'   [{rank+1}] {m["tag"]:7s} | {m["name"]:28s} | score={score:.4f}')

print('\n✅ Persistência verificada com sucesso.')

---
## Seção 11 — Resumo dos Artefatos Gerados

In [ ]:
# ============================================================
# RESUMO FINAL
# ============================================================

artifacts = [
    INDEX_PATH,
    METADATA_PATH,
    FIG_PATH,
    SIM_PATH,
]

print('=' * 60)
print('SPRINT 2 PLN — ARTEFATOS GERADOS')
print('=' * 60)
for p in artifacts:
    size = p.stat().st_size if p.exists() else 0
    unit = 'KB' if size < 1_000_000 else 'MB'
    size_str = f'{size/1024:.1f} {unit}' if size < 1_000_000 else f'{size/1024/1024:.2f} {unit}'
    print(f'   {"✅":2s} {p.name:40s} {size_str}')

print()
print('MÉTRICAS DE AVALIAÇÃO')
print('-' * 40)
for k in K_VALUES:
    print(f'   Mean Precision@{k} = {df_eval[f"P@{k}"].mean():.3f}')
    print(f'   Mean Recall@{k}    = {df_eval[f"R@{k}"].mean():.3f}')

print()
print('CONFIGURAÇÃO DO MODELO')
print('-' * 40)
print(f'   Modelo       : {MODEL_NAME}')
print(f'   Dimensão     : {EMBEDDING_DIM}')
print(f'   Índice FAISS : IndexFlatIP (coseno após norm L2)')
print(f'   N ativos     : {len(ASSETS)}')
print(f'   Sinônimos    : {sum(len(v) for v in SYNONYM_MAP.values())} variantes mapeadas')
print(f'   Templates    : base, status, sensores, tipo_extra, completo')
print(f'   Eval queries : {len(EVAL_SET)}')
print('=' * 60)

---
## Checklist de Entregáveis Sprint 2 PLN

| Requisito | Status |
|---|---|
| Identificar campos textuais disponíveis | ✅ |
| Geração de TAGs padronizadas | ✅ |
| Modelo `paraphrase-multilingual-MiniLM-L12-v2` | ✅ |
| Justificativa técnica do modelo | ✅ |
| Pipeline Texto → Embedding → Indexação | ✅ |
| Índice FAISS (`IndexFlatIP`, dim=384) | ✅ |
| Persistência do índice (`faiss_assets.index`) | ✅ |
| Módulo de consulta semântica | ✅ |
| Ranking por score de similaridade coseno | ✅ |
| Tratamento de sinônimos industriais | ✅ |
| Conjunto de 20 consultas de avaliação | ✅ |
| Precision@K (K=1,2,3) | ✅ |
| Recall@K (K=1,2,3) | ✅ |
| Gráficos de avaliação | ✅ |
| Matriz de similaridade entre ativos | ✅ |
| Templates de linguagem natural (5 tipos) | ✅ |
| Regras de composição documentadas | ✅ (ver `sprint2_pln_templates.md`) |
| Enriquecimento com TAG, temperatura, corrente, tensão, RPM, estado | ✅ |
| Dados da Sprint 1 reutilizados sem modificação | ✅ |